In [0]:
df=spark.read.format("delta")\
    .load("/Volumes/workspace/bronzes/bronzevolume/bookings/data")
display(df)

booking_id,passenger_id,flight_id,airport_id,amount,booking_date,_rescued_data
B00001,P0048,F0014,A048,850.72,2025-05-29,null
B00002,P0011,F0052,A003,376.63,2025-06-09,null
B00003,P0079,F0023,A012,534.02,2025-06-03,null
B00004,P0068,F0001,A039,1333.7,2025-06-16,null
B00005,P0189,F0019,A008,1334.96,2025-06-17,null
B00006,P0070,F0003,A004,296.13,2025-05-18,null
B00007,P0194,F0068,A016,460.14,2025-04-05,null
B00008,P0181,F0048,A017,1402.02,2025-06-04,null
B00009,P0096,F0081,A016,1444.51,2025-05-16,null
B00010,P0131,F0089,A034,292.39,2025-05-16,null


In [0]:
from pyspark.sql.functions import*
from pyspark.sql.types import*
df=df.withColumn("amount",col("amount").cast(DoubleType()))\
    .withColumn("upadated_timestamp",current_timestamp())\
    .withColumn("booking_date",to_date(col("booking_date")))\
    .drop("_rescued_data")
    

In [0]:
display(df.limit(5))

booking_id,passenger_id,flight_id,airport_id,amount,booking_date,upadated_timestamp
B01001,P0207,F0047,A004,912.64,2025-07-10,2026-03-06T20:27:29.952Z
B01002,P0124,F0088,A051,597.04,2025-07-01,2026-03-06T20:27:29.952Z
B01003,P0103,F0096,A036,724.88,2025-07-10,2026-03-06T20:27:29.952Z
B01004,P0049,F0069,A001,1370.99,2025-07-12,2026-03-06T20:27:29.952Z
B01005,P0210,F0084,A053,1476.15,2025-07-01,2026-03-06T20:27:29.952Z


In [0]:
@dlt.table(
    name="stage_bookings"
)
def stage_bookings():
    df=spark.readStream.format("delta")\
        .load("/Volumes/workspace/bronzes/bronzevolume/bookings/data")
    return df

In [0]:
@dlt.view(
    name="trans_bookings"
)
def trans_bookings():
    df=spark.readStream.table("stage_bookings")
    df=df.withColumn("amount",col("amount").cast(DounleType()))\
        .withColumn("updated_timestamp",current_timestamp)\
        .withColumn("booking_date",to_date(col("booking_date")))\
        .drop("_rescue_data")
    return df

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-6118766804254623>, line 1
----> 1 @dlt.view(
      2     name="trans_bookings"
      3 )
      4 def trans_bookings():
      5     df=spark.readStream.table("stage_bookings")
      6     df=df.withColumn("amount",col("amount").cast(DounleType()))\
      7         .withColumn("updated_timestamp",current_timestamp)\
      8         .withColumn("booking_date",to_date(col("booking_date")))\
      9         .drop("_rescue_data")

NameError: name 'dlt' is not defined

In [0]:
rules={
    "rule1":"booking_id is not null",
    "rule2":"passenger_id is not null"
}

In [0]:
@dlt.table(
    name="silver_bookings"
)
@dlt.expect_all(rules)
def silver_bookings():
    df=spark.readStream.table("trans_bookings")
    return df

In [0]:
df=spark.read.format("delta")\
         .load("/Volumes/workspace/bronzes/bronzevolume/flights/data/")

display(df)

flight_id,airline,origin,destination,flight_date,_rescued_data
F0001,Delta,Kellyfort,South Kathleen,2025-05-04,null
F0002,Qatar Airways,Lake Stephen,New Vincent,2025-04-29,null
F0003,Lufthansa,East Patrickborough,North Mary,2025-05-11,null
F0004,Delta,Maddenshire,Johnchester,2025-05-16,null
F0005,Qatar Airways,Bennettside,New Mistyhaven,2025-06-13,null
F0006,Air Canada,New Richardside,South Jamesborough,2025-05-16,null
F0007,Delta,Berryport,Miguelburgh,2025-05-24,null
F0008,Lufthansa,Briannachester,Cervantesland,2025-05-26,null
F0009,Delta,Alexandraborough,North Alexishaven,2025-06-10,null
F0010,Emirates,Kruegerchester,Martintown,2025-05-20,null


In [0]:
from pyspark.sql.functions import*
from pyspark.sql.types import*
df=df.withColumn("flight_date",to_date(col("flight_date"),"MM-dd-yyyy"))\
    .withColumn("modified_date",current_timestamp())\
    .drop("_rescued_data")

In [0]:
import dlt 
from pyspark.sql.functions import*
from pyspark.sql.types import*

@dlt.view(
    name="trans_flights"
)
def trans_flights():
    df=spark.readStream.format("delta")\
        .load("/Volumes/workspace/bronzes/bronzevolume/flights/data/")
    df=df.withColumn("flight_date",to_date(col("flight_date"),"MM-dd-yyyy"))\
    .withColumn("modified_date",current_timestamp())\
    .drop("_rescued_data")
    
    return df

In [0]:
from pyspark.sql.functions import*
from pyspark.sql.types import*

df=spark.read.format("delta").load("/Volumes/workspace/bronzes/bronzevolume/customers/data")\
    .withColumn("modified_date",current_timestamp())\
    .drop("_rescued_data")

display(df)

passenger_id,name,gender,nationality,modified_date
P0001,Kevin Ferguson,Male,Reunion,2026-03-10T17:00:40.427Z
P0002,Kathleen Martinez DVM,Female,Burkina Faso,2026-03-10T17:00:40.427Z
P0003,Cynthia Frazier,Male,Marshall Islands,2026-03-10T17:00:40.427Z
P0004,Ryan Ramsey,Male,Niger,2026-03-10T17:00:40.427Z
P0005,Mike Kim,Male,Taiwan,2026-03-10T17:00:40.427Z
P0006,Diana Adams,Male,Mayotte,2026-03-10T17:00:40.427Z
P0007,Sharon Moon,Male,Madagascar,2026-03-10T17:00:40.427Z
P0008,Cheryl Glenn,Male,Maldives,2026-03-10T17:00:40.427Z
P0009,Allen Lowery,Male,Rwanda,2026-03-10T17:00:40.427Z
P0010,Maria Medina,Male,Denmark,2026-03-10T17:00:40.427Z


In [0]:
from pyspark.sql.functions import*
from pyspark.sql.types import*
df=spark.read.format("delta").load("/Volumes/workspace/bronzes/bronzevolume/airports/data/")\
    .withColumn("modified_date",current_timestamp())\
        .drop("_rescued_data")
display(df)

airport_id,airport_name,city,country,modified_date
A001,Gregoryland International Airport,Kristenmouth,South Georgia and the South Sandwich Islands,2026-03-10T17:22:38.997Z
A002,East Kristin International Airport,North Michaelview,Bosnia and Herzegovina,2026-03-10T17:22:38.997Z
A003,Brownland International Airport,Samuelville,Costa Rica,2026-03-10T17:22:38.997Z
A004,Meghanton International Airport,Andrewsmouth,Macedonia,2026-03-10T17:22:38.997Z
A005,East Aaron International Airport,Davishaven,Monaco,2026-03-10T17:22:38.997Z
A006,Michaelburgh International Airport,East Blake,Iceland,2026-03-10T17:22:38.997Z
A007,West Jennifer International Airport,Jillianstad,Libyan Arab Jamahiriya,2026-03-10T17:22:38.997Z
A008,Port Craig International Airport,New Lisa,French Southern Territories,2026-03-10T17:22:38.997Z
A009,New Joshuafurt International Airport,Port Jamiehaven,Pitcairn Islands,2026-03-10T17:22:38.997Z
A010,Thompsontown International Airport,Murraychester,Ireland,2026-03-10T17:22:38.997Z


In [0]:
%sql
select * from workspace.silvers.silver_airports

airport_id,airport_name,city,country,modified_date
A001,Gregoryland International Airport,Kristenmouth,South Georgia and the South Sandwich Islands,2026-03-10T17:51:10.494Z
A002,East Kristin International Airport,North Michaelview,Bosnia and Herzegovina,2026-03-10T17:51:10.494Z
A003,Brownland International Airport,Samuelville,Costa Rica,2026-03-10T17:51:10.494Z
A004,Meghanton International Airport,Andrewsmouth,Macedonia,2026-03-10T17:51:10.494Z
A005,East Aaron International Airport,Davishaven,Monaco,2026-03-10T17:51:10.494Z
A006,Michaelburgh International Airport,East Blake,Iceland,2026-03-10T17:51:10.494Z
A007,West Jennifer International Airport,Jillianstad,Libyan Arab Jamahiriya,2026-03-10T17:51:10.494Z
A008,Port Craig International Airport,New Lisa,French Southern Territories,2026-03-10T17:51:10.494Z
A009,New Joshuafurt International Airport,Port Jamiehaven,Pitcairn Islands,2026-03-10T17:51:10.494Z
A010,Thompsontown International Airport,Murraychester,Ireland,2026-03-10T17:51:10.494Z


In [0]:
spark.read.format("delta")\
    .load("/Volumes/workspace/bronzes/bronzevolume/flights/data")\
    .show(5, truncate=False)

+---------+-------------+----------------+--------------+-----------+-------------+
|flight_id|airline      |origin          |destination   |flight_date|_rescued_data|
+---------+-------------+----------------+--------------+-----------+-------------+
|F0101    |Qatar Airways|Lake Joshuamouth|Johnshire     |2025-07-14 |NULL         |
|F0102    |Delta        |Schroederbury   |Sethfort      |2025-06-25 |NULL         |
|F0103    |Qatar Airways|Jameschester    |Ashleyberg    |2025-07-15 |NULL         |
|F0104    |Qatar Airways|Huangstad       |Catherinehaven|2025-06-29 |NULL         |
|F0105    |Air Canada   |Aprilton        |Victorville   |2025-07-05 |NULL         |
+---------+-------------+----------------+--------------+-----------+-------------+
only showing top 5 rows


In [0]:
from pyspark.sql.functions import to_date, col

df = spark.read.format("delta")\
    .load("/Volumes/workspace/bronzes/bronzevolume/flights/data/")

df.select("flight_date").show(5, truncate=False)

+-----------+
|flight_date|
+-----------+
|2025-07-14 |
|2025-06-25 |
|2025-07-15 |
|2025-06-29 |
|2025-07-05 |
+-----------+
only showing top 5 rows


In [0]:
df.printSchema()

root
 |-- flight_id: string (nullable = true)
 |-- airline: string (nullable = true)
 |-- origin: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- flight_date: string (nullable = true)
 |-- _rescued_data: string (nullable = true)



In [0]:
from pyspark.sql.functions import to_date, col

df = spark.read.format("delta")\
    .load("/Volumes/workspace/bronzes/bronzevolume/flights/data/")

df.withColumn("flight_date_parsed", to_date(col("flight_date"), "yyyy-MM-dd"))\
  .select("flight_date", "flight_date_parsed")\
  .show(5)

+-----------+------------------+
|flight_date|flight_date_parsed|
+-----------+------------------+
| 2025-07-14|        2025-07-14|
| 2025-06-25|        2025-06-25|
| 2025-07-15|        2025-07-15|
| 2025-06-29|        2025-06-29|
| 2025-07-05|        2025-07-05|
+-----------+------------------+
only showing top 5 rows


In [0]:
%sql
DROP TABLE IF EXISTS workspace.silvers.silver_business